In [ ]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

data_raw = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", data_raw.shape)
print(data_raw.dtypes)import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)
data_raw.head()

In [ ]:
data_raw = data_raw[
    data_raw["RainTomorrow"].notna()
].copy()

data_raw["RainTomorrow"] = (
    data_raw["RainTomorrow"]
    .str.strip()
    .map({
        "No": 0,
        "Yes": 1
    })
)

print("Shape after removing missing target:", data_raw.shape)

print("\nTarget distribution:")
print(data_raw["RainTomorrow"].value_counts())

print("\nMissing feature values:")
print(data_raw.isna().sum().sort_values(ascending=False))

classlabel = "RainTomorrow"

x_raw = data_raw.drop(columns=[classlabel])
y_raw = data_raw[classlabel]

print("Features:", x_raw.shape)
print("Target:", y_raw.shape)

In [ ]:
from sklearn.model_selection import train_test_split

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    x_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

print("Training:", x_train_raw.shape)
print("Testing:", x_test_raw.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

x_train = x_train_raw.copy()
x_test = x_test_raw.copy()

In [ ]:
x_train["Date"] = pd.to_datetime(
    x_train["Date"],
    errors="coerce"
)

x_test["Date"] = pd.to_datetime(
    x_test["Date"],
    errors="coerce"
)

reference_date = pd.Timestamp("2000-01-01")

x_train["Date"] = (
    x_train["Date"] - reference_date
).dt.days

x_test["Date"] = (
    x_test["Date"] - reference_date
).dt.days

#categorical features
categorical_cols = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print(categorical_cols)

category_maps = {}

for col in categorical_cols:

    categories = (
        x_train[col]
        .dropna()
        .unique()
        .tolist()
    )

    mapping = {
        value: number
        for number, value in enumerate(categories)
    }

    category_maps[col] = mapping

    x_train[col] = x_train[col].map(mapping)

    x_test[col] = x_test[col].map(mapping)

In [ ]:
print(x_train.dtypes.value_counts())

print("\nTraining missing values:")
print(x_train.isna().sum().sort_values(ascending=False))

print("\nTest missing values:")
print(x_test.isna().sum().sort_values(ascending=False))

print("\nTrain shape:", x_train.shape)
print("Test shape:", x_test.shape)

print(
    x_train.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()
)

In [ ]:
from skeLCS import eLCS
from sklearn.model_selection import train_test_split

x_sample, _, y_sample, _ = train_test_split(
    x_train,
    y_train,
    train_size=25000,
    random_state=42,
    stratify=y_train
)

raw_test_model = eLCS(
    learning_iterations=5000,
    N=1000,
    random_state=42
)

raw_test_model.fit(
    x_sample.values,
    y_sample.values
)

print("Raw eLCS test training complete")

In [ ]:
raw_test_predictions = raw_test_model.predict(
    x_test.values
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        raw_test_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test,
        raw_test_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        raw_test_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        raw_test_predictions
    )
)

In [ ]:
import sklearn
print("scikit-learn version:", sklearn.__version__)
from skeLCS import eLCS

raw_elcs = eLCS(
    learning_iterations=120000,
    N=7500,
    random_state=124655
)

raw_elcs.fit(
    x_train.values,
    y_train.values
)

print("Raw eLCS training complete")

raw_predictions = raw_elcs.predict(
    x_test.values
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        raw_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test,
        raw_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        raw_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        raw_predictions
    )
)

In [ ]:
import os

os.makedirs("../results", exist_ok=True)

raw_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "prediction": raw_predictions
})

raw_results.to_csv(
    "../results/raw_elcs_predictions.csv",
    index=False
)

print("Raw eLCS predictions saved")

raw_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "prediction": raw_predictions
})

raw_results.to_csv(
    "../results/raw_elcs_predictions.csv",
    index=False
)

print("Predictions saved")